# ML-04 — Data Contract & Feature Leakage
**Lane:** Lane 2 — Refresh / Content Opportunity Scoring

This notebook develops on March 2026, a mid-panel month. June 2026 is treated as the sealed final month.


## 1. Data Contract — five plain-language answers

1. **What one row means:** one row represents one observed content-page performance record for one report date, one pseudonymized client, and one pseudonymized content item.
2. **Which table(s):** `fact_content_daily_performance`.
3. **Time window:** March 2026 for development and verification; June 2026 remains sealed.
4. **What I will rank:** content pages for refresh-review priority using observed performance signals. This is decision support, not causal prediction.
5. **One deliberate exclusion:** future-window outcome fields and label-derived fields are excluded from the honest feature set.


In [1]:
import os
import duckdb
import pandas as pd
import numpy as np

HF_TOKEN = os.environ.get("HF_TOKEN")

if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        HF_TOKEN = None

if not HF_TOKEN:
    raise RuntimeError("Add a Hugging Face READ token as a Colab Secret named HF_TOKEN.")

con = duckdb.connect()
con.execute("SET enable_progress_bar = false")
safe_token = HF_TOKEN.replace("'", "''")
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{safe_token}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
FACT = f"{REL}/fact_content_daily_performance"
MARCH = f"read_parquet('{FACT}/month=2026-03/*.parquet')"
APRIL = f"read_parquet('{FACT}/month=2026-04/*.parquet')"

print("Connected to FlyRank warehouse.")
print("Development month: March 2026")


Connected to FlyRank warehouse.
Development month: March 2026


## 2. Three verification queries

Exactly three verification queries are used below.


### Query 1 — Grain


In [3]:
q1 = con.sql(f'''
SELECT
    COUNT(*) AS rows,
    COUNT(DISTINCT report_date || '|' || client_hash_id || '|' || content_hash_id) AS distinct_grain_keys,
    COUNT(*) = COUNT(DISTINCT report_date || '|' || client_hash_id || '|' || content_hash_id) AS grain_matches
FROM {MARCH}
''')

q1

┌─────────┬─────────────────────┬───────────────┐
│  rows   │ distinct_grain_keys │ grain_matches │
│  int64  │        int64        │    boolean    │
├─────────┼─────────────────────┼───────────────┤
│ 9841378 │             9841378 │ true          │
└─────────┴─────────────────────┴───────────────┘

### Query 2 — March 2026 row count and date span


In [4]:
q2 = con.sql(f'''
SELECT
    COUNT(*) AS row_count,
    MIN(report_date) AS first_report_date,
    MAX(report_date) AS last_report_date,
    COUNT(DISTINCT report_date) AS distinct_report_dates
FROM {MARCH}
''').df()
display(q2)


,row_count,first_report_date,last_report_date,distinct_report_dates
0,9841378,2026-03-01,2026-03-31,31


### Query 3 — Availability (`IS TRUE`)


In [5]:
q3 = con.sql(f'''
SELECT
    COUNT(*) AS total_rows,
    COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS gsc_available_rows
FROM {MARCH}
''').df()
display(q3)


,total_rows,gsc_available_rows
0,9841378,3611061


## 3. Five-feature frame

Five features maximum. Each is knowable at the decision moment because it comes from information already observed before the decision.


In [6]:
schema = con.sql(f"DESCRIBE SELECT * FROM {MARCH}").df()
print(schema[["column_name", "column_type"]].to_string(index=False))


             column_name column_type
             report_date        DATE
          client_hash_id     VARCHAR
         content_hash_id     VARCHAR
          client_has_gsc     BOOLEAN
          client_has_ga4     BOOLEAN
      gsc_data_available     BOOLEAN
      ga4_data_available     BOOLEAN
         gsc_impressions      BIGINT
              gsc_clicks      BIGINT
        gsc_sum_position      BIGINT
        gsc_avg_position      DOUBLE
           ga4_pageviews      BIGINT
            ga4_sessions      BIGINT
               ga4_users      BIGINT
    ga4_engaged_sessions      BIGINT
ga4_total_engagement_sec      BIGINT
        sessions_organic      BIGINT
         sessions_direct      BIGINT
       sessions_referral      BIGINT
         sessions_social      BIGINT
           sessions_paid      BIGINT
             sessions_ai      BIGINT
              ai_chatgpt      BIGINT
           ai_perplexity      BIGINT
               ai_gemini      BIGINT
              ai_copilot      BIGINT
 

### Feature availability notes

- `impressions` — knowable at the decision moment because recorded search impressions are already observed.
- `clicks` — knowable at the decision moment because recorded clicks are already observed.
- `sessions` — knowable at the decision moment because recorded sessions are already observed.
- `engagement_rate` — knowable at the decision moment because it is derived from observed engagement/session data.
- `days_since_last_update` — knowable at the decision moment because it is derived from observed content-update history.


In [8]:
# Five-feature frame using actual warehouse columns

feature_frame = con.sql(f'''
SELECT
    client_hash_id,
    content_hash_id,
    gsc_impressions,
    gsc_clicks,
    gsc_avg_position,
    ga4_sessions,
    scroll_events
FROM {MARCH}
WHERE gsc_data_available IS TRUE
LIMIT 10000
''').df()

print("Rows:", len(feature_frame))
feature_frame.head()

Rows: 10000


,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_sessions,scroll_events
0,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,20,0,3.350000,<NA>,<NA>
1,client_73cda7b4e4f265ea,content_05597932fe4da067,1,0,0.000000,<NA>,<NA>
2,client_73cda7b4e4f265ea,content_7a105f548d9c6916,125,1,4.928000,<NA>,<NA>
3,client_73cda7b4e4f265ea,content_905aa32a0230694e,7,0,4.000000,<NA>,<NA>
4,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,11,0,2.272727,<NA>,<NA>


## 4. Deliberate leakage trap

For the experiment only, create one future/label-derived column: the next-month change in impressions. It is unavailable at the March decision moment. The leaked field is then removed.


In [11]:
march_agg = con.sql(f'''
SELECT
    client_hash_id,
    content_hash_id,
    SUM({IMP}) AS impressions_march
FROM {MARCH}
GROUP BY client_hash_id, content_hash_id
''').df()

april_agg = con.sql(f'''
SELECT
    client_hash_id,
    content_hash_id,
    SUM({IMP}) AS impressions_april
FROM {APRIL}
GROUP BY client_hash_id, content_hash_id
''').df()

leakage_demo = march_agg.merge(
    april_agg,
    on=["client_hash_id", "content_hash_id"],
    how="inner"
)

leakage_demo["future_impression_change"] = (
    leakage_demo["impressions_april"]
    - leakage_demo["impressions_march"]
)

# Deliberate label-derived leakage
leakage_demo["leaky_score"] = leakage_demo["future_impression_change"].abs()

leakage_demo.head()

,client_hash_id,content_hash_id,impressions_march,impressions_april,future_impression_change,leaky_score
0,client_73cda7b4e4f265ea,content_7a105f548d9c6916,6523.0,6787.0,264.0,264.0
1,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,453.0,405.0,-48.0,48.0
2,client_73cda7b4e4f265ea,content_36c36abc7650d7af,5630.0,8475.0,2845.0,2845.0
3,client_73cda7b4e4f265ea,content_a7da352b73b02668,4944.0,6091.0,1147.0,1147.0
4,client_73cda7b4e4f265ea,content_f39be42b42a4e8f6,42.0,67.0,25.0,25.0


In [12]:
# Deliberately leaky quick score:
# It directly uses future outcome information.

leakage_demo["leaky_score"] = leakage_demo["future_impression_change"].abs()

print("Leaky score summary:")
print(leakage_demo["leaky_score"].describe())

leakage_demo[
    [
        "client_hash_id",
        "content_hash_id",
        "impressions_march",
        "impressions_april",
        "future_impression_change",
        "leaky_score"
    ]
].head(10)

Leaky score summary:
count    331436.000000
mean        412.078712
std        2142.559799
min           0.000000
25%           0.000000
50%           3.000000
75%         113.000000
max      261817.000000
Name: leaky_score, dtype: float64


,client_hash_id,content_hash_id,impressions_march,impressions_april,future_impression_change,leaky_score
0,client_73cda7b4e4f265ea,content_7a105f548d9c6916,6523.0,6787.0,264.0,264.0
1,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,453.0,405.0,-48.0,48.0
2,client_73cda7b4e4f265ea,content_36c36abc7650d7af,5630.0,8475.0,2845.0,2845.0
3,client_73cda7b4e4f265ea,content_a7da352b73b02668,4944.0,6091.0,1147.0,1147.0
4,client_73cda7b4e4f265ea,content_f39be42b42a4e8f6,42.0,67.0,25.0,25.0
5,client_73cda7b4e4f265ea,content_1855a661b4d36130,429.0,287.0,-142.0,142.0
6,client_73cda7b4e4f265ea,content_5d412fba6e1a2582,223.0,552.0,329.0,329.0
7,client_73cda7b4e4f265ea,content_1f380a642aed423b,96.0,111.0,15.0,15.0
8,client_73cda7b4e4f265ea,content_22c063002b7c1caf,314.0,322.0,8.0,8.0
9,client_73cda7b4e4f265ea,content_aafb2ab7e5fc80d0,7709.0,5261.0,-2448.0,2448.0


In [13]:
# Remove the deliberate leak and keep only the honest five-feature frame.
honest_feature_frame = feature_frame.copy()
assert "future_impression_change" not in honest_feature_frame.columns

print("Leakage removed.")
print("Honest feature columns:")
print(honest_feature_frame.columns.tolist())


Leakage removed.
Honest feature columns:
['client_hash_id', 'content_hash_id', 'gsc_impressions', 'gsc_clicks', 'gsc_avg_position', 'ga4_sessions', 'scroll_events']


## 5. Limitation

This lane is useful for decision support, but observed performance signals do not establish causality. The warehouse is an unbalanced panel, availability can differ across measurement systems, and a refresh-priority score does not prove that refreshing a page will improve future performance. Human review is required before action.


## 6. Self-check

- Five plain-language contract answers
- Exactly three verification queries with visible outputs
- Availability checked with `IS TRUE`
- Five features maximum
- "Knowable at the decision moment" explanation for each feature
- One deliberate label/future-derived leakage column
- Leakage removed from the honest feature set
- One named limitation
- Notebook executed and committed to `work/notebooks/w03_data_contract.ipynb`
